**-setup**

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS gold_External
URL 'abfss://gold@retailprojec1data.dfs.core.windows.net/'
WITH(
    STORAGE CREDENTIAL retail_project_cred
);

In [0]:

from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg, min as spark_min, 
    max as spark_max, round as spark_round, 
    date_trunc, current_timestamp, lit, when,
    concat, first
)

# Widgets
dbutils.widgets.text("storage_account", "retailprojec1data")
dbutils.widgets.text("silver_container", "silver")
dbutils.widgets.text("gold_container", "gold")

storage_account = dbutils.widgets.get("storage_account")
silver_container = dbutils.widgets.get("silver_container")
gold_container = dbutils.widgets.get("gold_container")

base_silver = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net"
base_gold = f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net"

# Silver source paths
fact_path = f"{base_silver}/fact_transactions/"
inventory_path = f"{base_silver}/current_inventory/"
dim_products_path = f"{base_silver}/dim_products/"

# Gold target paths
daily_revenue_path = f"{base_gold}/daily_revenue_by_store/"
weekly_categories_path = f"{base_gold}/top_categories_weekly/"
clv_path = f"{base_gold}/customer_lifetime_value/"
inventory_risk_path = f"{base_gold}/inventory_at_risk/"

print("Silver sources:")
print(f"  Fact:       {fact_path}")
print(f"  Inventory:  {inventory_path}")
print(f"  Products:   {dim_products_path}")
print(f"\nGold targets:")
print(f"  Revenue:    {daily_revenue_path}")
print(f"  Categories: {weekly_categories_path}")
print(f"  CLV:        {clv_path}")
print(f"  Inv Risk:   {inventory_risk_path}")

**-Load silver tables**

In [0]:
fact = spark.read.format("delta").load(fact_path)
print(f"Fact transactions: {fact.count()}")

# Filter to completed transactions only (exclude refunded/voided for revenue)
completed = fact.filter(col("status") == "completed")
print(f"Completed transactions: {completed.count()}")

**-Gold Table1-Daily revenue by Store**

In [0]:
daily_revenue = (completed
    .groupBy(
        "store_id",
        "store_location",
        "transaction_date"
    )
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        count("*").alias("transaction_count"),
        spark_round(avg("total_amount"), 2).alias("avg_transaction_value"),
        spark_round(spark_sum("gross_margin"), 2).alias("total_gross_margin"),
        spark_round(spark_sum("quantity"), 0).alias("units_sold"),
    )
    .withColumn("gold_ingestion_ts", current_timestamp())
    .orderBy("transaction_date", "store_id")
)

print(f"Daily revenue rows: {daily_revenue.count()}")
daily_revenue.show(10, truncate=False)

# Write to Gold
(daily_revenue.write
    .format("delta")
    .mode("overwrite")
    .save(daily_revenue_path))

print(f"✅ Gold daily_revenue_by_store written")

**-Gold Table2:Top Categories weekly**

In [0]:
weekly_categories = (completed
    .withColumn("week_start", date_trunc("week", col("transaction_date")))
    .groupBy(
        "week_start",
        "category"
    )
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(spark_sum("quantity"), 0).alias("units_sold"),
        count("*").alias("transaction_count"),
        spark_round(avg("total_amount"), 2).alias("avg_transaction_value"),
        spark_round(spark_sum("gross_margin"), 2).alias("total_gross_margin"),
    )
    .withColumn("gold_ingestion_ts", current_timestamp())
    .orderBy("week_start", col("total_revenue").desc())
)

print(f"Weekly categories rows: {weekly_categories.count()}")
weekly_categories.show(15, truncate=False)

# Write to Gold
(weekly_categories.write
    .format("delta")
    .mode("overwrite")
    .save(weekly_categories_path))

print(f"Gold top_categories_weekly written")

**-Gold Table-3:Custmer Life time Value**

In [0]:
clv = (completed
    .groupBy(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "loyalty_tier",
        "customer_city",
        "customer_state"
    )
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("total_spent"),
        count("*").alias("transaction_count"),
        spark_round(avg("total_amount"), 2).alias("avg_transaction_value"),
        spark_min("transaction_date").alias("first_purchase"),
        spark_max("transaction_date").alias("last_purchase"),
        spark_round(spark_sum("gross_margin"), 2).alias("total_margin_generated"),
        spark_round(spark_sum("quantity"), 0).alias("total_units_purchased"),
    )
    .withColumn("customer_name", 
        concat(col("customer_first_name"), lit(" "), col("customer_last_name")))
    .withColumn("gold_ingestion_ts", current_timestamp())
    .orderBy(col("total_spent").desc())
)

print(f"CLV rows: {clv.count()}")
print("\nTop 10 customers by total spend:")
clv.select(
    "customer_name", "loyalty_tier", "customer_city",
    "total_spent", "transaction_count", "avg_transaction_value",
    "first_purchase", "last_purchase"
).show(10, truncate=False)

# Write to Gold
(clv.write
    .format("delta")
    .mode("overwrite")
    .save(clv_path))

print(f" Gold customer_lifetime_value written")

**-Gold Table-4: Inventory at risk**

In [0]:
inventory = spark.read.format("delta").load(inventory_path)
dim_products = spark.read.format("delta").load(dim_products_path)

# Get current product names
current_products = dim_products.filter(col("is_current") == True).select("sku", "product_name", "category")

inventory_risk = (inventory
    .join(current_products, "sku", "left")
    .withColumn("stock_status",
        when(col("stock_on_hand") <= 0, "OUT_OF_STOCK")
        .when(col("stock_on_hand") < col("reorder_point") * 0.5, "CRITICAL")
        .when(col("stock_on_hand") < col("reorder_point"), "LOW")
        .otherwise("HEALTHY")
    )
    .withColumn("units_below_reorder",
        when(col("stock_on_hand") < col("reorder_point"),
            col("reorder_point") - col("stock_on_hand"))
        .otherwise(lit(0))
    )
    .select(
        "store_id",
        "sku",
        "product_name",
        "category",
        "stock_on_hand",
        "reorder_point",
        "units_below_reorder",
        "stock_status",
        "snapshot_date"
    )
    .withColumn("gold_ingestion_ts", current_timestamp())
    .orderBy("stock_status", "units_below_reorder")
)

print(f"Inventory risk rows: {inventory_risk.count()}")

print("\n🔴 Critical and low stock items:")
inventory_risk.filter(col("stock_status").isin("CRITICAL", "LOW", "OUT_OF_STOCK")).show(20, truncate=False)

print("\nStock status summary:")
inventory_risk.groupBy("stock_status").count().show()

# Write to Gold
(inventory_risk.write
    .format("delta")
    .mode("overwrite")
    .save(inventory_risk_path))

print(f" Gold inventory_at_risk written")

**-exit**

In [0]:
dbutils.notebook.exit("gold_layer_complete")